[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/practice/advanced/03_trustworthy_ml_and_causality.ipynb)

# Luyện tập nâng cao 3 — ML đáng tin cậy và suy luận nhân quả

Hiệu chỉnh xác suất, khoảng dự đoán có bảo đảm, ước lượng tác động nhân quả, công bằng giữa các nhóm, giám sát trôi — tự cài và kiểm chứng trên dữ liệu có "sự thật".

Chương liên quan: 36, 37, 38, 40.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/practice/advanced

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_causal_marketing

rng = np.random.default_rng(0)

**Bài 1 — Platt scaling từ đầu.** Cho điểm số `s` (chưa hiệu chỉnh) và nhãn trên tập hiệu chỉnh, tìm $a, b$ tối đa likelihood của $p = \sigma(a s + b)$ bằng **Newton–Raphson** (bắt đầu $a = b = 0$, 25 vòng). `fit_platt(s, y)` → `(a, b)`; so với `LogisticRegression(C=1e10)` trên cột `s`.

In [ ]:
def fit_platt(s, y, n_iter=25):
    ### START CODE HERE ### (≈ 9 dòng)
    # STARTER: return 1.0, 0.0
    A = np.column_stack([s, np.ones_like(s)])
    theta = np.zeros(2)
    for _ in range(n_iter):
        p = 1 / (1 + np.exp(-A @ theta))
        grad = A.T @ (p - y)
        hess = A.T @ (A * (p * (1 - p))[:, None]) + 1e-9 * np.eye(2)
        theta -= np.linalg.solve(hess, grad)
    return float(theta[0]), float(theta[1])
    ### END CODE HERE ###


true_logit = rng.normal(size=5000)
y1 = (rng.random(5000) < 1 / (1 + np.exp(-true_logit))).astype(int)
scores = 0.3 * true_logit + 0.5                                         # điểm thô: xếp hạng đúng nhưng sai thang
a, b = fit_platt(scores, y1)
ref = LogisticRegression(C=1e10).fit(scores[:, None], y1)
print(f"a = {a:.3f}, b = {b:.3f} | scikit-learn: {ref.coef_[0, 0]:.3f}, {ref.intercept_[0]:.3f}")

In [ ]:
check_close([a, b], [ref.coef_[0, 0], ref.intercept_[0]], tol=1e-3, message="khớp hồi quy logistic")
check_close([a, b], [1 / 0.3, -0.5 / 0.3], tol=0.4, message="khôi phục gần đúng phép biến đổi thật")

**Bài 2 — Expected Calibration Error.** `ece(p, y, n_bins=10)`: bin đều trên [0, 1]; ECE = $\sum_b \frac{n_b}{n}\,|\bar y_b - \bar p_b|$. Tính ECE của điểm thô `sigmoid(scores)` và sau Platt.

In [ ]:
def ece(p, y, n_bins=10):
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: return 0.0
    p, y = np.asarray(p, float), np.asarray(y, float)
    bins = np.minimum((p * n_bins).astype(int), n_bins - 1)
    total = 0.0
    for k in range(n_bins):
        m = bins == k
        if m.any():
            total += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(total)
    ### END CODE HERE ###


raw_p = 1 / (1 + np.exp(-scores))
platt_p = 1 / (1 + np.exp(-(a * scores + b)))
print(f"ECE thô: {ece(raw_p, y1):.3f} | sau Platt: {ece(platt_p, y1):.3f}")

In [ ]:
check_close(ece([0.2, 0.2, 0.8, 0.8], [0, 1, 1, 1]), 0.5 * 0.3 + 0.5 * 0.2, message="ví dụ nhỏ")
check_greater(ece(raw_p, y1), 0.05, "điểm thô lệch")
check_less(ece(platt_p, y1), 0.02, "Platt hiệu chỉnh tốt")

**Bài 3 — Conformalized quantile regression (CQR).** Khoảng dự đoán thích nghi với nhiễu **không đều**: huấn luyện hai mô hình phân vị 5% và 95% (`GradientBoostingRegressor(loss="quantile", alpha=…, random_state=0)`) trên phần train; trên phần hiệu chỉnh tính điểm $E_i = \max(\hat q_{lo}(x_i) - y_i,\; y_i - \hat q_{hi}(x_i))$; $Q$ = phần tử thứ $\lceil (n+1)(1-\alpha)\rceil$ nhỏ nhất của $E$ (α = 0.1); khoảng test $[\hat q_{lo} - Q, \hat q_{hi} + Q]$. Đo `coverage` và độ rộng trung bình ở vùng nhiễu nhỏ / lớn (`width_low_noise`, `width_high_noise`).

In [ ]:
x3 = rng.uniform(0, 10, 3000)
y3 = np.sin(x3) + rng.normal(0, 0.1 + 0.15 * x3, 3000)                 # nhiễu tăng theo x
X3 = x3[:, None]
idx_tr, idx_cal, idx_te = np.arange(1500), np.arange(1500, 2200), np.arange(2200, 3000)
### START CODE HERE ### (≈ 9 dòng)
q_lo = GradientBoostingRegressor(loss="quantile", alpha=0.05, random_state=0).fit(X3[idx_tr], y3[idx_tr])
q_hi = GradientBoostingRegressor(loss="quantile", alpha=0.95, random_state=0).fit(X3[idx_tr], y3[idx_tr])
E = np.maximum(q_lo.predict(X3[idx_cal]) - y3[idx_cal], y3[idx_cal] - q_hi.predict(X3[idx_cal]))
k = int(np.ceil((len(E) + 1) * 0.9 - 1e-9))
Q = np.sort(E)[k - 1]
lower, upper = q_lo.predict(X3[idx_te]) - Q, q_hi.predict(X3[idx_te]) + Q
coverage = float(np.mean((y3[idx_te] >= lower) & (y3[idx_te] <= upper)))
width_low_noise = float(np.mean((upper - lower)[x3[idx_te] < 3]))
width_high_noise = float(np.mean((upper - lower)[x3[idx_te] > 7]))
### END CODE HERE ###
print(f"độ phủ {coverage:.1%} | độ rộng TB: x < 3: {width_low_noise:.2f}, x > 7: {width_high_noise:.2f}")

In [ ]:
check_true(0.86 <= coverage <= 0.95, "độ phủ ≈ 90%")
check_greater(width_high_noise, 2.5 * width_low_noise, "khoảng thích nghi: rộng hơn ở vùng nhiều nhiễu")

**Bài 4 — IPW và AIPW.** Dữ liệu quan sát `make_causal_marketing(20000, seed=4)` (coupon phát không ngẫu nhiên, có cột hiệu ứng thật). Ước lượng ATE bằng: chênh lệch ngây thơ `naive`; IPW Hajek với điểm xu hướng logistic (cắt [0.02, 0.98]) `ipw`; AIPW với mô hình kết quả tuyến tính riêng cho từng nhóm `aipw`.

In [ ]:
data4 = make_causal_marketing(20000, seed=4)
X4 = data4[["age", "income_mvnd", "past_purchases"]].values
T4, Y4 = data4.coupon.values, data4.spend_kvnd.values
true_ate = data4.true_effect.mean()
### START CODE HERE ### (≈ 9 dòng)
naive = Y4[T4 == 1].mean() - Y4[T4 == 0].mean()
e = np.clip(LogisticRegression(max_iter=1000).fit(X4, T4).predict_proba(X4)[:, 1], 0.02, 0.98)
w1, w0 = T4 / e, (1 - T4) / (1 - e)
ipw = (w1 * Y4).sum() / w1.sum() - (w0 * Y4).sum() / w0.sum()
mu1 = LinearRegression().fit(X4[T4 == 1], Y4[T4 == 1]).predict(X4)
mu0 = LinearRegression().fit(X4[T4 == 0], Y4[T4 == 0]).predict(X4)
aipw = np.mean(mu1 - mu0 + T4 * (Y4 - mu1) / e - (1 - T4) * (Y4 - mu0) / (1 - e))
### END CODE HERE ###
print(f"ATE thật {true_ate:.1f} | ngây thơ {naive:.1f} | IPW {ipw:.1f} | AIPW {aipw:.1f}")

In [ ]:
check_greater(abs(naive - true_ate), 50, "so sánh ngây thơ chệch nặng (confounding)")
check_less(abs(ipw - true_ate), 15, "IPW gần ATE thật")
check_less(abs(aipw - true_ate), 8, "AIPW gần ATE thật")

**Bài 5 — Chỉ số công bằng.** Mô hình duyệt hồ sơ cho hai nhóm A, B. Tính: chênh lệch tỉ lệ duyệt (demographic parity difference) `dp_diff` = P(duyệt | A) − P(duyệt | B); chênh lệch tỉ lệ duyệt **trong số người thực sự đủ điều kiện** (equal opportunity difference) `eo_diff`. Hai chỉ số kể hai câu chuyện khác nhau thế nào?

In [ ]:
group = np.where(rng.random(10000) < 0.5, "A", "B")
qualified = (rng.random(10000) < np.where(group == "A", 0.6, 0.3)).astype(int)    # tỉ lệ đủ điều kiện thật khác nhau
score = qualified * 1.0 + rng.normal(0, 0.6, 10000) - 0.2 * (group == "B")       # mô hình có thiên lệch nhẹ với B
approved = score > 0.5
### START CODE HERE ### (≈ 2 dòng)
dp_diff = approved[group == "A"].mean() - approved[group == "B"].mean()
eo_diff = approved[(group == "A") & (qualified == 1)].mean() - approved[(group == "B") & (qualified == 1)].mean()
### END CODE HERE ###
print(f"chênh lệch tỉ lệ duyệt: {dp_diff:+.3f} | chênh lệch duyệt trong số người đủ điều kiện: {eo_diff:+.3f}")

In [ ]:
check_greater(dp_diff, 0.2, "tỉ lệ duyệt chênh nhiều — phần lớn do tỉ lệ đủ điều kiện thật khác nhau")
check_true(0.03 < eo_diff < 0.2, "trong số người đủ điều kiện, B vẫn bị duyệt ít hơn — dấu hiệu thiên lệch của mô hình")

Demographic parity đòi tỉ lệ duyệt bằng nhau **bất kể** tỉ lệ đủ điều kiện thật — ở đây chênh lệch lớn chủ yếu vì hai nhóm khác nhau thật. Equal opportunity so sánh **trong số người đủ điều kiện** — nó cô lập phần bất công do mô hình (người B đủ điều kiện vẫn bị từ chối nhiều hơn). Các tiêu chí công bằng thường **không thể** đồng thời thoả mãn khi tỉ lệ nền khác nhau (Kleinberg et al., 2016) — chọn tiêu chí là quyết định chính sách, phải nói rõ trong model card (chương 36).

**Bài 6 — Giám sát: PSI cho biến phân loại và cảnh báo.** Viết `psi_categorical(ref, cur, eps=1e-4)` (tính cả danh mục mới), rồi `drift_alerts(ref_df, cur_df, threshold=0.25)` → danh sách tên cột (phân loại) có PSI vượt ngưỡng.

In [ ]:
def psi_categorical(ref, cur, eps=1e-4):
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: return 0.0
    ref, cur = pd.Series(ref), pd.Series(cur)
    cats = sorted(set(ref) | set(cur), key=str)
    r = np.clip(ref.value_counts(normalize=True).reindex(cats, fill_value=0).values, eps, None)
    c = np.clip(cur.value_counts(normalize=True).reindex(cats, fill_value=0).values, eps, None)
    return float(np.sum((c - r) * np.log(c / r)))
    ### END CODE HERE ###


def drift_alerts(ref_df, cur_df, threshold=0.25):
    ### START CODE HERE ### (≈ 1 dòng)
    # STARTER: return []
    return [col for col in ref_df.columns if psi_categorical(ref_df[col], cur_df[col]) > threshold]
    ### END CODE HERE ###


ref6 = pd.DataFrame({"channel": rng.choice(["app", "web", "store"], 5000, p=[0.5, 0.3, 0.2]),
                     "region": rng.choice(["N", "S", "C"], 5000, p=[0.4, 0.4, 0.2])})
cur6 = pd.DataFrame({"channel": rng.choice(["app", "web", "store", "partner"], 5000, p=[0.3, 0.2, 0.1, 0.4]),
                     "region": rng.choice(["N", "S", "C"], 5000, p=[0.41, 0.39, 0.2])})
print({c: round(psi_categorical(ref6[c], cur6[c]), 3) for c in ref6}, drift_alerts(ref6, cur6))

In [ ]:
check_close(psi_categorical(["a", "b"] * 50, ["a", "b"] * 50), 0.0, message="giống hệt → 0")
check_equal(drift_alerts(ref6, cur6), ["channel"], "chỉ kênh bán (có danh mục mới 'partner') bị cảnh báo")